# Combining the tables

Each secondary table is aggregated to one row per applicant (`src/aggregate.py`: mean, max, min and sum of numeric columns, share of each category, row count) and joined onto `application_train` on `SK_ID_CURR`. Tables are added **one at a time** and we keep the 5-fold CV AUC after each, so every table has to earn its place.

Same model and folds in every step.

The first code cell below loads the cleaned application table and the pre-aggregated secondary tables.

In [1]:
# Setup: load the application table, apply cleaning and features, and load the pre-aggregated tables
import sys
sys.path.append("..")  # so src/ can be imported from the notebooks folder

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score

from src.features import clean, add_features

df = pd.read_csv("../data/raw/HC_application_train.csv")
y = df["TARGET"]
base = add_features(clean(df.drop(columns="TARGET")))  # application table with cleaning and engineered features

# Secondary tables already aggregated to one row per applicant (index = SK_ID_CURR)
tables = {name: pd.read_pickle(f"../data/processed/{name}.pkl")
          for name in ["bureau", "previous", "installments"]}
{name: t.shape for name, t in tables.items()}  # (applicants, aggregated columns) per table

{'bureau': (305811, 124),
 'previous': (338857, 224),
 'installments': (339587, 37)}

**Insight:** Aggregated tables loaded: `bureau` has 305,811 rows and 124 columns, `previous` 338,857 and 224, `installments` 339,587 and 37. `previous` and `installments` have more rows than the 307,511 training applicants, so they probably also contain ids outside the training file.

Not every applicant appears in every table (for example, no previous credit card). After a left join those applicants get missing values, which LightGBM handles.

The next cell checks how many applicants actually appear in each table.

In [2]:
# Coverage check: what share of applicants has at least one record in each table
for name, t in tables.items():
    covered = df["SK_ID_CURR"].isin(t.index).mean() * 100
    print(f"{name}: {covered:.1f}% of applicants have records")

bureau: 85.7% of applicants have records
previous: 94.6% of applicants have records
installments: 94.8% of applicants have records


**Insight:** Coverage is high everywhere: 85.7% of applicants have bureau records and about 95% have previous-application or installment records. Bureau has the most gaps, so its columns will be missing for roughly one applicant in seven.

Now we add the tables one at a time and re-score after each join. This shows how much each source contributes, so a table that adds columns but no AUC would be easy to spot and drop.

In [3]:
# Add the tables one at a time and record the 5-fold CV AUC after each join
def cv_auc(X):
    X = X.drop(columns="SK_ID_CURR").copy()  # the ID is a key, not a feature
    # LightGBM reads text columns natively once they are typed as category
    for c in X.select_dtypes(include=["object", "string"]):
        X[c] = X[c].astype("category")
    # colsample_bytree=0.3 samples 30% of columns per tree, which suits the wide feature set
    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, colsample_bytree=0.3,
                               random_state=42, verbose=-1)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # same folds in every step
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    return scores.mean(), scores.std(), X.shape[1]

# Step 0: application features only
X = base
mean, std, n = cv_auc(X)
results = [{"added": "application only (with features)", "features": n, "AUC mean": round(mean, 4), "AUC std": round(std, 4)}]
print(results[-1])

# Steps 1-3: left join each table on SK_ID_CURR (applicants without records get NaN) and re-score
for name, t in tables.items():
    X = X.join(t, on="SK_ID_CURR")
    mean, std, n = cv_auc(X)
    results.append({"added": "+ " + name, "features": n, "AUC mean": round(mean, 4), "AUC std": round(std, 4)})
    print(results[-1])

pd.DataFrame(results)  # summary table, one row per step

{'added': 'application only (with features)', 'features': 133, 'AUC mean': np.float64(0.7669), 'AUC std': np.float64(0.0049)}


{'added': '+ bureau', 'features': 257, 'AUC mean': np.float64(0.773), 'AUC std': np.float64(0.0043)}


{'added': '+ previous', 'features': 481, 'AUC mean': np.float64(0.7819), 'AUC std': np.float64(0.0044)}


{'added': '+ installments', 'features': 518, 'AUC mean': np.float64(0.7868), 'AUC std': np.float64(0.0039)}


,added,features,AUC mean,AUC std
0,application only (with features),133,0.7669,0.0049
1,+ bureau,257,0.7730,0.0043
2,+ previous,481,0.7819,0.0044
3,+ installments,518,0.7868,0.0039


**Insight:** Each table raises mean AUC: 0.7669 (application only) to 0.7730 with bureau (+0.0061), 0.7819 with previous (+0.0089) and 0.7868 with installments (+0.0049), about +0.020 in total. The fold std is 0.004 to 0.005, so the previous-application step is clearly above noise while the bureau and installments steps are only about one to one and a half fold stds, a modest but consistent gain since the folds are identical. The application-only score here (0.7669) differs from notebook 03 (0.7650) because this model also uses `colsample_bytree=0.3`.